In [0]:
-- 1. Dimensão Clientes
CREATE OR REPLACE TABLE gold.dim_clientes AS
SELECT DISTINCT
  customer_id AS id_cliente,
  customer_unique_id AS id_cliente_unico,
  cidade,
  estado
FROM silver.customers;

SELECT * FROM gold.dim_clientes;

In [0]:
-- 2. Dimensão Produtos
CREATE OR REPLACE TABLE gold.dim_produtos AS
SELECT DISTINCT
  product_id AS id_produto,
  categoria_produto
FROM silver.products;

SELECT * FROM gold.dim_produtos;

In [0]:
-- 3. Dimensão Avaliações Enriquecidas por IA (1a avaliação mais recente por pedido para manter relação 1:N)
CREATE OR REPLACE TABLE gold.dim_avaliacoes_ia AS
WITH avaliacoes_deduplicadas AS (
  SELECT
    order_id AS id_pedido,
    review_id AS id_avaliacao,
    nota_avaliacao,
    sentimento,
    categoria_problema,
    resumo_curto,
    comentario,
    ROW_NUMBER() OVER (PARTITION BY order_id ORDER BY data_avaliacao DESC) AS rn  -- OVER anuncia Window Function
  FROM silver.reviews_enriquecidas_ia
)
SELECT
  id_pedido,
  id_avaliacao,
  nota_avaliacao,
  sentimento,
  categoria_problema,
  resumo_curto,
  comentario
FROM avaliacoes_deduplicadas
WHERE rn = 1;
/* Leitura:
CREATE TABLE ... AS: WITH ... AS [CTE], SELECT ... FROM [CTE] WHERE ...
Crie uma tabela A contendo, a partir do conjunto de dados temporário B, as colunas tal a partir do conjunto B onde C.
*/

SELECT * FROM gold.dim_avaliacoes_ia;

In [0]:
-- 3.1 Trim da coluna de categoria e macrocategoria: Excel extraído da query anterior
CREATE OR REPLACE TABLE gold.dim_produtos AS
WITH base_limpa AS (
  SELECT DISTINCT
    product_id AS id_produto,
    CASE
      WHEN categoria_produto = 'eletrodomesticos 2' THEN 'Eletrodomesticos'

      WHEN categoria_produto = 'casa conforto 2' THEN 'Casa Conforto'

      WHEN categoria_produto = 'pcs' THEN 'PCs'

      WHEN categoria_produto = 'pc gamer' THEN 'PC Gamer'

      WHEN categoria_produto = 'market place' THEN 'Marketplace'

      WHEN categoria_produto = 'dvds blu ray' THEN 'DVDs Blu-ray'

      WHEN categoria_produto = 'cds dvds musicais' THEN 'CDs E DVDs Musicais'

      ELSE INITCAP(categoria_produto)
    END AS subcategoria
  FROM silver.products
) -- From here: solicitação ao Gemini para criar macrocategorias
SELECT
  id_produto,
  subcategoria,
  CASE
    WHEN subcategoria IN (
      'Cama Mesa Banho', 'Moveis Decoracao', 'Utilidades Domesticas', 'Moveis Escritorio',
      'Casa Conforto', 'Moveis Cozinha Area De Servico Jantar E Jardim', 'Moveis Sala',
      'Moveis Quarto', 'Moveis Colchao E Estofado', 'La Cuisine', 'Flores',
      'Artigos De Natal', 'Artigos De Festas'
    ) THEN 'Casa, Móveis e Decoração'

    WHEN subcategoria IN (
      'Construcao Ferramentas Construcao', 'Construcao Ferramentas Ferramentas', 'Ferramentas Jardim', 'Construcao Ferramentas Seguranca',
      'Construcao Ferramentas Jardim', 'Casa Construcao', 'Construcao Ferramentas Iluminacao',
      'Sinalizacao E Seguranca', 'Construcao Ferramentas'
    ) THEN 'Construção e Ferramentas'

    WHEN subcategoria IN (
      'Eletronicos', 'Telefonia', 'Consoles Games', 'Informatica Acessorios',
      'Telefonia Fixa', 'Audio', 'PCs', 'Cine Foto', 'Tablets Impressao Imagem', 'PC Gamer'
    ) THEN 'Tecnologia e Eletrônicos'

    WHEN subcategoria IN (
      'Fashion Roupa Masculina', 'Malas Acessorios', 'Fashion Bolsas E Acessorios',
      'Fashion Underwear E Moda Praia', 'Fashion Calcados', 'Fashion Roupa Feminina',
      'Fashion Roupa Infanto Juvenil', 'Fashion Esporte', 'Relogios Presentes'
    ) THEN 'Moda e Acessórios'

    WHEN subcategoria IN (
      'Eletrodomesticos', 'Eletroportateis', 'Portateis Cozinha E Preparadores De Alimentos',
      'Climatizacao', 'Portateis Casa Forno E Cafe'
    ) THEN 'Eletrodomésticos e Portáteis'

    WHEN subcategoria IN ('Beleza Saude', 'Perfumaria') THEN 'Beleza e Saúde'

    WHEN subcategoria IN ('Bebes', 'Fraldas Higiene') THEN 'Bebês e Higiene'

    WHEN subcategoria IN ('Esporte Lazer', 'Brinquedos', 'Cool Stuff') THEN 'Esporte, Lazer e Brinquedos'

    WHEN subcategoria IN (
      'Livros Tecnicos', 'Papelaria', 'Livros Interesse Geral', 'Livros Importados',
      'DVDs Blu-ray', 'Artes', 'Artes E Artesanato', 'Musica', 'Instrumentos Musicais',
      'CDs E DVDs Musicais'
    ) THEN 'Livros, Papelaria e Cultura'

    WHEN subcategoria IN ('Alimentos', 'Bebidas', 'Alimentos Bebidas') THEN 'Alimentos e Bebidas'

    WHEN subcategoria = 'Automotivo' THEN 'Automotivo'

    WHEN subcategoria = 'Pet Shop' THEN 'Pet Shop'

    WHEN subcategoria IN (
      'Agro Industria E Comercio', 'Industria Comercio E Negocios',
      'Marketplace', 'Seguros E Servicos'
    ) THEN 'Indústria, Comércio e Serviços'

    ELSE 'Não Categorizado'
  END AS macro_categoria
FROM base_limpa;

SELECT * FROM gold.dim_produtos;

In [0]:
-- 4. Dimensão Calendário (gerada dinamicamente a partir do intervalo de compras)
CREATE OR REPLACE TABLE gold.dim_calendario AS
WITH limites AS (
  SELECT
    MIN(CAST(data_compra AS DATE)) AS data_min,
    MAX(CAST(data_compra AS DATE)) AS data_max
  FROM silver.orders
),
datas AS (
  SELECT EXPLODE(SEQUENCE(data_min, data_max, INTERVAL 1 DAY)) AS data
  FROM limites
)
SELECT
  data,
  YEAR(data) AS ano,
  CASE MONTH(data)
    WHEN 1  THEN 'Jan'
    WHEN 2  THEN 'Fev'
    WHEN 3  THEN 'Mar'
    WHEN 4  THEN 'Abr'
    WHEN 5  THEN 'Mai'
    WHEN 6  THEN 'Jun'
    WHEN 7  THEN 'Jul'
    WHEN 8  THEN 'Ago'
    WHEN 9  THEN 'Set'
    WHEN 10 THEN 'Out'
    WHEN 11 THEN 'Nov'
    WHEN 12 THEN 'Dez'
  END AS mes_abv,
  DATE_FORMAT(data, 'MMM/yy') AS mes_ano,
  QUARTER(data) AS trimestre,
  WEEKOFYEAR(data) AS semana,
  DAYOFMONTH(data) AS dia_mes,
  DAYOFWEEK(data) AS dia_semana_num
FROM datas;

SELECT * FROM gold.dim_calendario;

In [0]:
-- 1. Tabela fato única que junta orders, order_items e dim_avaliacoes_ia
CREATE OR REPLACE TABLE gold.fact_vendas AS
SELECT
  i.order_id AS id_pedido,
  i.order_item_id AS id_item_pedido,
  o.customer_id AS id_cliente,
  i.product_id AS id_produto,
  CAST(o.data_compra AS DATE) AS data_compra,
  o.order_status AS status_pedido,
  o.status_sla,
  o.dias_diferenca_entrega,
  i.valor_produto,
  i.valor_frete,
  (i.valor_produto + i.valor_frete) AS valor_total_item,
  CASE WHEN ia.id_pedido IS NOT NULL THEN 1 ELSE 0 END AS flag_avaliado_ia
FROM silver.order_items i
INNER JOIN silver.orders o
  ON i.order_id = o.order_id
LEFT JOIN gold.dim_avaliacoes_ia ia
  ON i.order_id = ia.id_pedido;

SELECT * FROM gold.fact_vendas LIMIT 500;